In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/relay.py
/kaggle/input/comp

In [2]:
import re
import math
from pathlib import Path
from typing import Optional, Tuple

import pandas as pd
import sympy as sp
from sympy import Integer
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication_application,
)

TRANSFORMS = standard_transformations + (implicit_multiplication_application,)


# -------------------------
# Utilities / Normalization
# -------------------------
def normalize_latex(s: str) -> str:
    """Normaliza construções LaTeX para uma string compatível com sympy/python."""
    if not isinstance(s, str):
        return ""
    t = s

    # remove ambientes LaTeX e comentários
    t = re.sub(r"\\begin\{.*?\}.*?\\end\{.*?\}", "", t, flags=re.S)
    t = re.sub(r"%.+", "", t)

    # substituições básicas
    t = t.replace(r"\cdot", "*").replace(r"\times", "*")
    t = t.replace(r"\,", "").replace(r"\;", "").replace(r"\:", "")
    t = t.replace(r"\left", "").replace(r"\right", "")
    t = t.replace(r"\ ", "")
    t = t.replace(r"\;", "")

    # frações \frac{a}{b} -> (a)/(b)
    t = re.sub(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", r"(\1)/(\2)", t)

    # sqrt e raiz enésima
    t = re.sub(r"\\sqrt\{([^{}]+)\}", r"sqrt(\1)", t)
    t = re.sub(r"\\sqrt\[(\d+)\]\{([^{}]+)\}", r"root(\2,\1)", t)

    # piso/teto
    t = t.replace(r"\lfloor", "floor(").replace(r"\rfloor", ")")
    t = t.replace(r"\lceil", "ceiling(").replace(r"\rceil", ")")

    # valor absoluto |x| -> abs(x) (não-guloso)
    t = re.sub(r"\|([^|]+?)\|", r"abs(\1)", t)

    # porcentagem x\% -> (x/100)
    t = re.sub(r"([0-9]+)\\\%", r"(\1/100)", t)

    # potência: ^{...} e ^x (tratamento seguro)
    t = re.sub(r"\^\{([^{}]+)\}", lambda m: "**(" + m.group(1) + ")", t)
    t = re.sub(r"\^([A-Za-z0-9_]+)", lambda m: "**" + m.group(1), t)

    # remove delimitadores $...$
    t = t.replace("$", "")

    # normaliza espaços
    t = re.sub(r"\s+", " ", t).strip()
    return t


# -------------------------
# Math expression extraction
# -------------------------
def extract_math_expr(text: str) -> str:
    """
    Extrai expressão matemática contígua de um texto.
    Remove palavras comuns (Compute, Find, etc) no início.
    """
    if not isinstance(text, str):
        return ""
    
    original = text
    # Remove palavras introdutórias comuns no início
    text = re.sub(r"^(Compute|Find|Calculate|Determine|What is|What are|Evaluate|Simplify|Solve)\s+", "", text, flags=re.IGNORECASE)
    
    # Tenta encontrar padrões com LaTeX ou expressões matemáticas
    # Procura por estruturas envolvidas
    patterns = [
        r"\\lfloor.+?\\rfloor",
        r"\\lceil.+?\\rceil",
        r"\$([^$]+)\$",  # Conteúdo entre $...$
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE | re.DOTALL)
        if match:
            result = match.group(1) if "$" in pattern else match.group(0)
            return result.strip()
    
    # Se não encontrou padrão especial, retorna o texto processado
    return text.strip() if text.strip() else original


# -------------------------
# Sympy parsing wrapper
# -------------------------
def parse_sympy(s: str):
    """Parse (possivelmente LaTeX) para expressão sympy."""
    s2 = normalize_latex(s)
    if not s2:
        return Integer(0)
    try:
        return parse_expr(s2, transformations=TRANSFORMS, evaluate=True)
    except Exception:
        # tentativa alternativa substituindo ^ por **
        try:
            s3 = s2.replace("^", "**")
            return parse_expr(s3, transformations=TRANSFORMS, evaluate=True)
        except Exception:
            return Integer(0)


# -------------------------
# Detecção e avaliação modular
# -------------------------
def detect_mod_pattern(text: str) -> Optional[Tuple[str, int]]:
    """
    Detecta padrões:
      - 'remainder when <expr> is divided by <n>'
      - '<expr> mod 10^5' ou 'modulo 100000'
      - '<expr> mod n'
    Retorna (expr, mod) ou None.
    """
    if not isinstance(text, str):
        return None

    m = re.search(
        r"remainder\s+when\s+(.+?)\s+is\s+divided\s+by\s+([0-9]+)",
        text,
        flags=re.I,
    )
    if m:
        expr = m.group(1).strip()
        modn = int(m.group(2))
        return expr, modn

    # Padrão: <expr> mod 10^5 (com space ou sem)
    m2 = re.search(r"([\d\^{}\-+*/.\\]+)\s+mod(?:ulo)?\s+10\^?5\b", text, flags=re.I)
    if m2:
        expr = m2.group(1).strip()
        return expr, 10**5

    # Padrão: <expr> mod n (números simples)
    m3 = re.search(r"([\d\^{}\-+*/.\\]+)\s+mod(?:ulo)?\s+([0-9]+)\b", text, flags=re.I)
    if m3:
        expr = m3.group(1).strip()
        modn = int(m3.group(2))
        return expr, modn

    return None


def apply_modular_eval(expr_str: str, modn: int) -> int:
    """
    Avalia expr_str módulo modn de forma eficiente quando possível.
    Detecta padrões simples de potência para usar pow(base, exp, mod).
    """
    expr_str = extract_math_expr(expr_str)
    s = normalize_latex(expr_str)

    # detectar padrão base**exp no final (exp inteiro simples)
    m_pow = re.search(r"^([0-9]+)\*\*\(?([0-9]+)\)?", s)
    if m_pow:
        base_s = m_pow.group(1).strip()
        exp_s = m_pow.group(2).strip()
        try:
            base_val = int(base_s)
            exp_val = int(exp_s)
            result = pow(base_val, exp_val, modn)
            return result
        except Exception:
            pass

    # fallback: avaliar inteiro completo e aplicar mod
    try:
        val = parse_sympy(s)
        if isinstance(val, sp.Expr) and getattr(val, "is_integer", False):
            ival = int(sp.Integer(val))
            return ival % modn
        ival = int(sp.Integer(val))
        return ival % modn
    except Exception:
        try:
            ival = eval_safe(s)
            return int(ival) % modn
        except Exception:
            return 0


# -------------------------
# Avaliação segura
# -------------------------
def eval_safe(expr: str, verbose: bool = False):
    """
    Avalia expressão usando sympy preferencialmente.
    Se falhar, usa eval Python com ambiente restrito.
    Verbose mode mostra qual estratégia foi usada.
    """
    if not isinstance(expr, str) or expr.strip() == "":
        return 0

    local_dict = {
        "sqrt": sp.sqrt,
        "floor": sp.floor,
        "ceiling": sp.ceiling,
        "abs": sp.Abs,
        "root": lambda x, n: sp.root(parse_sympy(str(x)), Integer(int(n))),
    }

    mod_detect = detect_mod_pattern(expr)
    if mod_detect:
        expr_part, modn = mod_detect
        if verbose:
            print(f"[eval_safe] Using modular evaluation: {expr_part} mod {modn}")
        return apply_modular_eval(expr_part, modn)

    # Tenta extrair a expressão matemática se houver texto adicional
    math_expr = extract_math_expr(expr)

    # Estratégia 1: Sympy com normalização LaTeX
    try:
        normalized = normalize_latex(math_expr)
        sym = parse_expr(normalized, local_dict=local_dict, transformations=TRANSFORMS, evaluate=True)
        if isinstance(sym, sp.Integer) or (hasattr(sym, "is_integer") and sym.is_integer):
            result = int(sp.Integer(sym))
            if verbose:
                print(f"[eval_safe] Strategy 1 (sympy integer): '{math_expr}' -> {result}")
            return result
        if isinstance(sym, sp.Rational):
            if sym.q == 1:
                result = int(sym.p)
            else:
                result = int(float(sym))
            if verbose:
                print(f"[eval_safe] Strategy 1 (sympy rational): '{math_expr}' -> {result}")
            return result
        try:
            result = int(sp.N(sym))
            if verbose:
                print(f"[eval_safe] Strategy 1 (sympy N): '{math_expr}' -> {result}")
            return result
        except Exception:
            result = int(float(sym))
            if verbose:
                print(f"[eval_safe] Strategy 1 (sympy float): '{math_expr}' -> {result}")
            return result
    except Exception as e1:
        if verbose:
            print(f"[eval_safe] Strategy 1 failed: {e1}")
    
    # Estratégia 2: Avaliação segura com eval restrito
    try:
        safe_globals = {"__builtins__": {}}
        safe_locals = {
            "math": math,
            "sqrt": math.sqrt,
            "floor": math.floor,
            "ceil": math.ceil,
            "abs": abs,
            "pow": pow,
        }
        safe_expr = normalize_latex(math_expr).replace("^", "**")
        result = int(eval(safe_expr, safe_globals, safe_locals))
        if verbose:
            print(f"[eval_safe] Strategy 2 (safe eval): '{math_expr}' -> {result}")
        return result
    except Exception as e2:
        if verbose:
            print(f"[eval_safe] Strategy 2 failed: {e2}")
    
    # Estratégia 3: Extrair número mais significativo (não apenas o primeiro)
    # Tenta encontrar números que pareçam ser respostas (maiores, no final, etc)
    numbers = re.findall(r"-?\d+(?:\.\d+)?", math_expr)
    if numbers:
        if verbose:
            print(f"[eval_safe] Strategy 3 (fallback number extraction): found {numbers}")
        # Se há apenas um número, retorna ele
        if len(numbers) == 1:
            try:
                result = int(float(numbers[0]))
                if verbose:
                    print(f"[eval_safe] Strategy 3: single number -> {result}")
                return result
            except Exception:
                pass
        # Se há múltiplos números, tenta ordem aritmética simples
        # Por exemplo, em "1-1", tenta avaliar manualmente
        try:
            # Tenta substituir números diretamente
            evaluated = 0
            if "-" in math_expr and len(numbers) >= 2:
                try:
                    result = int(float(numbers[0])) - int(float(numbers[1])) if "-" in math_expr else int(float(numbers[0]))
                    if verbose:
                        print(f"[eval_safe] Strategy 3 (simple arithmetic): {numbers[0]} - {numbers[1]} -> {result}")
                    return result
                except Exception:
                    pass
        except Exception:
            pass
        
        # Fallback final: retorna o maior número encontrado
        try:
            result = int(float(max([float(n) for n in numbers])))
            if verbose:
                print(f"[eval_safe] Strategy 3 (max number): {result}")
            return result
        except Exception:
            pass
    
    if verbose:
        print(f"[eval_safe] All strategies failed for '{expr}'")
    return 0


# -------------------------
# Solver principal
# -------------------------
def solve_problem_text(text: str, force_mod_100000: bool = False, verbose: bool = False) -> int:
    """
    Heurística para resolver enunciados:
      - detecta requisitos modulares e avalia com aritmética modular quando possível
      - avalia via sympy ou fallback seguro
      - retorna inteiro em 0..99999 (aplica modulo 100000 no final)
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 0

    try:
        mod_detect = detect_mod_pattern(text)
        if mod_detect:
            expr_part, modn = mod_detect
            if verbose:
                print(f"[DEBUG] Detected mod pattern: mod={modn}, expr='{expr_part}'")
            val = apply_modular_eval(expr_part, modn)
        else:
            val = eval_safe(text, verbose=verbose)

        try:
            val = int(val)
        except Exception:
            val = int(float(val))

        if force_mod_100000:
            val = val % 100000

        val = int(val) % 100000
        if verbose:
            print(f"[solve_problem_text] Final result: {val}")
        return val
    except Exception as e:
        if verbose:
            print(f"[solve_problem_text] Exception: {e}")
        return 0


# -------------------------
# Geração de submissão
# -------------------------
def generate_submission(
    test_csv_path: str,
    sample_submission_path: Optional[str] = None,
    out_path: str = "submission.csv",
    verbose: bool = False,
):
    """
    Lê test CSV, detecta colunas id e texto do problema, calcula previsões e escreve CSV de submissão.
    """
    test_df = pd.read_csv(test_csv_path)

    # detecta coluna id
    id_col = None
    for c in ["id", "problem_id", "index", "test_id"]:
        if c in test_df.columns:
            id_col = c
            break
    if id_col is None:
        id_col = test_df.columns[0]

    # detecta coluna de texto do problema
    prob_col = None
    for c in ["problem", "question", "text", "latex", "problem_text"]:
        if c in test_df.columns:
            prob_col = c
            break
    if prob_col is None:
        for c in test_df.columns:
            if test_df[c].dtype == object and c != id_col:
                prob_col = c
                break
    if prob_col is None:
        test_df["problem_text"] = ""
        prob_col = "problem_text"

    preds = []
    for idx, row in test_df.iterrows():
        txt = str(row[prob_col]) if pd.notna(row[prob_col]) else ""
        try:
            p = solve_problem_text(txt, force_mod_100000=False, verbose=verbose)
        except Exception:
            p = 0
        p = int(p) % 100000
        preds.append(p)

    if sample_submission_path and Path(sample_submission_path).exists():
        sample = pd.read_csv(sample_submission_path)
        out = sample.copy()
        ans_col = None
        for c in sample.columns[1:]:
            ans_col = c
            break
        if ans_col is None:
            ans_col = "answer"
            out[ans_col] = 0
        out[ans_col] = preds
        out[sample.columns[0]] = test_df[id_col].values
        out.to_csv(out_path, index=False)
    else:
        out = pd.DataFrame({id_col: test_df[id_col].values, "answer": preds})
        out.to_csv(out_path, index=False)

    print(f"Wrote {out_path} with {len(preds)} predictions.")


# -------------------------
# Testes rápidos / exemplos
# -------------------------
def _run_unit_tests():
    tests = [
        ("Compute \\lfloor 10^4 \\sqrt{2} \\rfloor", 14142),
        ("Find 3^{2025} mod 10^5", 29443),
        ("What is the remainder when 12345 is divided by 1000?", 345),
        ("\\frac{1}{2} + \\frac{1}{2}", 1),
        ("| -5 |", 5),
        ("What is $1-1$?", 0),
        ("What is $0\\times10$?", 0),
    ]
    ok = True
    for expr, expected in tests:
        got = solve_problem_text(expr, verbose=False)
        if got != expected:
            ok = False
            print(f"[TEST FAIL] '{expr}' -> got {got}, expected {expected}")
        else:
            print(f"[TEST OK] '{expr}' -> {got}")
    if ok:
        print("All unit tests passed.")
    else:
        print("Some unit tests failed.")


# -------------------------
# CLI / Exemplo de uso
# -------------------------
if __name__ == "__main__":
    # Caminhos locais (desenvolvimento)
    LOCAL_TEST = "test.csv"
    LOCAL_SAMPLE = "sample_submission.csv"
    LOCAL_OUT = "submission.csv"
    
    # Caminhos do Kaggle
    KAGGLE_TEST = "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv"
    KAGGLE_SAMPLE = "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv"
    KAGGLE_OUT = "/kaggle/working/submission.csv"


    # Detecta ambiente e escolhe caminhos
    if Path(KAGGLE_TEST).exists():
        TEST_CSV = KAGGLE_TEST
        SAMPLE_SUB = KAGGLE_SAMPLE
        OUT = KAGGLE_OUT
    else:
        TEST_CSV = LOCAL_TEST
        SAMPLE_SUB = LOCAL_SAMPLE
        OUT = LOCAL_OUT

    print("Running quick unit tests...")
    _run_unit_tests()

    if Path(TEST_CSV).exists():
        generate_submission(TEST_CSV, SAMPLE_SUB, OUT, verbose=True)
    else:
        print(f"Arquivo not found: {TEST_CSV}")


Running quick unit tests...
[TEST OK] 'Compute \lfloor 10^4 \sqrt{2} \rfloor' -> 14142
[TEST OK] 'Find 3^{2025} mod 10^5' -> 29443
[TEST OK] 'What is the remainder when 12345 is divided by 1000?' -> 345
[TEST OK] '\frac{1}{2} + \frac{1}{2}' -> 1
[TEST OK] '| -5 |' -> 5
[TEST OK] 'What is $1-1$?' -> 0
[TEST OK] 'What is $0\times10$?' -> 0
All unit tests passed.
[eval_safe] Strategy 1 (sympy integer): '1-1' -> 0
[solve_problem_text] Final result: 0
[eval_safe] Strategy 1 (sympy integer): '0\times10' -> 0
[solve_problem_text] Final result: 0
[eval_safe] Strategy 1 failed: invalid syntax (<string>, line 1)
[eval_safe] Strategy 2 failed: invalid syntax (<string>, line 1)
[eval_safe] Strategy 3 (fallback number extraction): found ['4', '4']
[eval_safe] Strategy 3 (max number): 4
[solve_problem_text] Final result: 4
Wrote /kaggle/working/submission.csv with 3 predictions.
